# 02_03 combined — index `time_df` + roadmap to full ETL

This notebook is for **review and reproducibility**: one place for the **authoritative** `time_df` rules and a short map of what still lives in the original full notebook.

| File | Role (unchanged on disk) |
|------|--------------------------|
| `02_03_Feature_space.ipynb` | Full pipeline: DB extracts, cohort loads, micro/AST prep, vitals, demographics, **LLM / mapper inputs**, exploratory cells. |
| `02_03_Feature_space_clean.ipynb` | Earlier timing-only extract; logic is **folded here** so you maintain a single timing definition. |
| **This notebook** | **Only** the cohort timing table `time_df` you should feed into `02_4_*` (plus documentation). |

## Index-culture / `time_df` rules (methods)

1. **Cohort AST** — restrict `all_ast_df` to respiratory cohort `hadm_id` (`resp_ast_df`).
2. **First ICU clock** — `first_icu_admit_time` = minimum `icu_admit_time` across all ICU rows for that `hadm_id`.
3. **Exclude whole admission** if **any** AST in `resp_ast_df` has `charttime` **strictly before** `first_icu_admit_time` (no pre–first-ICU cultures on the index admission).
4. **Index time** — among remaining admissions, keep rows with `charttime >= first_icu_admit_time` and take the **earliest** `charttime` per `hadm_id` → `first_ast_time`.
5. **ICU stay alignment** — if `first_ast_time` lies inside `[icu_admit_time, icu_discharge_time]` for some stay, attach that `stay_id` and ICU times; if it falls in a ward gap between stays, `stay_id` stays empty and `ast_inside_icu_stay=False` (timing still uses `first_icu_admit_time` for `hours_icu_to_ast` when unmatched).

**Output:** `hh.parq(..., "time_df_after_first_icu")` → use this parquet in `02_4_Feature_space_creation_clean.ipynb` instead of legacy `time_df*.parquet`.

**Prior admissions (features):** In `02_4`, `get_prior_events` uses `subject_id` and events **strictly before** index `hospital_admit_time` — prior `hadm_id`s only.


## 1. Setup

Adjust parquet paths if your extract filenames change.


In [ ]:
import pandas as pd
from pathlib import Path

import tools.helpers as hh

DATA_ROOT = Path("/Users/gnaanikko.pa/Documents/Academic /MIMIC/model_building")
PARQ_DIR = DATA_ROOT / "parq"

# Inputs: same sources as `02_03_Feature_space.ipynb` cohort + AST export
RESP_COHORT_PATH = PARQ_DIR / "resp_inf_cohort_23Jan26_1953.parquet"
ALL_AST_PATH = PARQ_DIR / "all_ast_df27Feb26_1751.parquet"


## 2. Load cohort and AST

- `resp_inf_cohort` supplies ICU intervals and hospital times.
- `all_ast_df` is filtered to cohort admissions; **index logic uses `charttime`** (sample time), not `storetime` (result time).


In [ ]:
resp_inf_cohort = pd.read_parquet(RESP_COHORT_PATH)
all_ast_df = pd.read_parquet(ALL_AST_PATH)

resp_ast_df = hh.df_subset(all_ast_df, resp_inf_cohort, by_col="hadm_id").copy()
resp_ast_df["charttime"] = pd.to_datetime(resp_ast_df["charttime"])

icu_stays_df = resp_inf_cohort[
    ["hadm_id", "stay_id", "icu_admit_time", "icu_discharge_time"]
].drop_duplicates()

print("resp_ast_df:", resp_ast_df.shape, "| icu stay rows:", len(icu_stays_df))


## 3. Build `time_df` (authoritative)

All steps are in **one** cell so diffs stay easy to review. Comments mark each rule.


In [ ]:
# --- Rule: first ICU admit clock per hospital admission ---
first_icu_admit = (
    icu_stays_df.groupby("hadm_id", as_index=False)["icu_admit_time"]
    .min()
    .rename(columns={"icu_admit_time": "first_icu_admit_time"})
)

ra_all = resp_ast_df.merge(first_icu_admit, on="hadm_id")

# --- Rule: drop entire hadm_id if ANY culture precedes first ICU ---
hadm_any_pre_icu_ast = (
    ra_all.loc[ra_all["charttime"] < ra_all["first_icu_admit_time"], "hadm_id"]
    .drop_duplicates()
)
print("Excluded hadm_id (any AST before first ICU):", len(hadm_any_pre_icu_ast))

# --- Rule: first index culture = earliest charttime on/after first ICU ---
eligible = ra_all[
    (~ra_all["hadm_id"].isin(hadm_any_pre_icu_ast))
    & (ra_all["charttime"] >= ra_all["first_icu_admit_time"])
].copy()

idx = eligible.groupby("hadm_id")["charttime"].idxmin()
first_rows = eligible.loc[idx].reset_index(drop=True).rename(
    columns={"charttime": "first_ast_time"}
)

# --- Rule: match ICU stay that contains first_ast_time (nth stay allowed) ---
ast_expanded = first_rows[["hadm_id", "first_ast_time"]].merge(icu_stays_df, on="hadm_id")
ast_expanded["inside"] = (
    (ast_expanded["first_ast_time"] >= ast_expanded["icu_admit_time"])
    & (ast_expanded["first_ast_time"] <= ast_expanded["icu_discharge_time"])
)
containing = (
    ast_expanded[ast_expanded["inside"]]
    .sort_values(["hadm_id", "icu_admit_time"])
    .groupby("hadm_id", as_index=False)
    .first()
)
containing = containing[["hadm_id", "stay_id", "icu_admit_time", "icu_discharge_time"]].rename(
    columns={
        "stay_id": "matched_stay_id",
        "icu_admit_time": "icu_admit_time_matched",
        "icu_discharge_time": "icu_discharge_time_matched",
    }
)

time_df = first_rows.merge(containing, on="hadm_id", how="left")
time_df["ast_inside_icu_stay"] = time_df["matched_stay_id"].notna()
time_df["stay_id"] = time_df["matched_stay_id"]
time_df["icu_admit_time"] = time_df["icu_admit_time_matched"]
time_df["icu_discharge_time"] = time_df["icu_discharge_time_matched"]

# Hospital-level anchors: one row per hadm (first ICU row order for stable demo fields)
hadm_demo = (
    resp_inf_cohort.sort_values(["hadm_id", "icu_admit_time"])
    .groupby("hadm_id", as_index=False)
    .first()[
        [
            "subject_id",
            "hadm_id",
            "hospital_admit_time",
            "hospital_discharge_time",
            "hospital_death_time",
        ]
    ]
)

time_df = time_df.drop(
    columns=[
        "matched_stay_id",
        "icu_admit_time_matched",
        "icu_discharge_time_matched",
    ],
    errors="ignore",
)
time_df = time_df.merge(hadm_demo, on=["hadm_id", "subject_id"], how="left")

# Hours: use matched ICU admit when inside a stay; else first_icu_admit_time
icu_ref_admit = time_df["icu_admit_time"].fillna(time_df["first_icu_admit_time"])
time_df["hours_adm_to_ast"] = (
    time_df["first_ast_time"] - time_df["hospital_admit_time"]
).dt.total_seconds() / 3600
time_df["hours_icu_to_ast"] = (
    time_df["first_ast_time"] - icu_ref_admit
).dt.total_seconds() / 3600
time_df["hours_first_icu_admit_to_ast"] = (
    time_df["first_ast_time"] - time_df["first_icu_admit_time"]
).dt.total_seconds() / 3600

out_cols = [
    "subject_id",
    "hadm_id",
    "stay_id",
    "hospital_admit_time",
    "hospital_discharge_time",
    "hospital_death_time",
    "icu_admit_time",
    "icu_discharge_time",
    "first_icu_admit_time",
    "first_ast_time",
    "ast_inside_icu_stay",
    "hours_adm_to_ast",
    "hours_icu_to_ast",
    "hours_first_icu_admit_to_ast",
]
time_df = time_df[[c for c in out_cols if c in time_df.columns]]

print("time_df rows:", len(time_df))
print("ast_inside_icu_stay:", int(time_df["ast_inside_icu_stay"].sum()), "True")


## 4. Save and inspect

`hh.parq` appends a timestamp — point `02_4_*` at the newest `time_df_after_first_icu_*.parquet`.


In [ ]:
hh.dxx(time_df)
hh.parq(time_df, "time_df_after_first_icu")


## 5. Full ETL (`02_03_Feature_space.ipynb`) — do not duplicate here

Run the **original** notebook when you need:

- Postgres extracts (microbiology, chartevents, admissions, etc.)
- Building or refreshing `all_ast_df`, `all_micro_*`, combined columns for **LLM input**
- Vitals pulls, demographics, exploratory tables
- Any **mapper / `AST_PATTERN` / export** for the external LLM run

**Do not** use legacy `time_df05Mar26_2130.parquet`-style files for new analyses if this combined notebook’s output is your methods definition — merge `02_4` on the **`time_df_after_first_icu_*`** file from section 4.


## 6. Optional — print latest saved timing parquet

Uncomment to copy-paste into `02_4`.


In [ ]:
# candidates = sorted(PARQ_DIR.glob("time_df_after_first_icu*.parquet"), key=lambda p: p.stat().st_mtime, reverse=True)
# print("Latest:", candidates[0] if candidates else "(none)")
